#Databricks notebook  - **Squad 3 | Batch**  
###Conecta ao ADLS Gen2 via Service Principal e lista os arquivos disponíveis. 

In [0]:
%pip install azure-storage-file-datalake azure-identity pandas python-dotenv
dbutils.library.restartPython()

In [0]:
# Carregamento do arquivo .env e Conexão com o ADLS
import os
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# Carrega as variáveis do arquivo .env localizado na mesma pasta do notebook
load_dotenv()

# Captura as variáveis de ambiente
CLIENT_ID       = os.getenv("CLIENT_ID")
CLIENT_SECRET   = os.getenv("CLIENT_SECRET")
TENANT_ID       = os.getenv("TENANT_ID")
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
CONTAINER       = os.getenv("CONTAINER")

# Validação simples para garantir que o arquivo .env foi lido
if not all([CLIENT_ID, CLIENT_SECRET, TENANT_ID, STORAGE_ACCOUNT, CONTAINER]):
    raise ValueError("❌ Erro: Algumas variáveis não foram encontradas no arquivo .env. Verifique o arquivo.")

print("✅ Excelente...variáveis carregadas com sucesso via arquivo .env!")
print(f"   Conectando ao Storage: {STORAGE_ACCOUNT} -> Container: {CONTAINER}\n")

# 1. Autenticação via Service Principal
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

# 2. Inicialização do Cliente do Data Lake
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential
)

# 3. Foco no Container desejado (Raw/Bronze)
filesystem_client = service_client.get_file_system_client(CONTAINER)

# 4. Listagem dos arquivos na pasta da camada Raw
print(f"📂 Arquivos em '{CONTAINER}/batch-data':\n")
try:
    for item in filesystem_client.get_paths(path="batch-data"):
        tipo = "📁" if item.is_directory else "📄"
        print(f"  {tipo} {item.name}")
except Exception as e:
    print(f"❌ Erro ao listar os arquivos: {e}")
